In [1]:
import json

def strip_to_ens(obj, parent_key=None):
    """
    Strip everything except 'ens' data.
    - Retain the top-level 'mean' and 'std' branches (since they have their own ensembles).
    - Drop quartiles (nq*) and local 'mean' entries inside extra_annual.
    """
    if isinstance(obj, dict):
        new_obj = {}
        for k, v in obj.items():
            # Drop quartiles and local means
            if k.startswith("nq") or (k == "mean" and parent_key == "extra_annual"):
                continue
            # Recurse into mean/std branches
            if k in ("mean", "std"):
                new_obj[k] = strip_to_ens(v, parent_key=k)
            elif k == "ens":
                new_obj[k] = v
            else:
                new_obj[k] = strip_to_ens(v, parent_key=k)
        return new_obj
    elif isinstance(obj, list):
        return [strip_to_ens(v, parent_key=parent_key) for v in obj]
    else:
        return obj

path = "inputdata_TS_12/v20230301/WaterCyclePanels/"
file = "Hydro_vars_change_20210201_derived_extra"
ext = ".json"

with open(path + file + ext) as f:
    data = json.load(f)

with open(path + file + "_raw" + ext, "w") as f:
    json.dump(strip_to_ens(data), f, indent=2)
